<a href="https://colab.research.google.com/github/dickypps/mkb-ml-rul-motor/blob/main/mkb-ml-rul-motor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

url = 'https://raw.githubusercontent.com/dickypps/mkb-ml-rul-motor/main/data/raw.csv'
df = pd.read_csv(url)

print(df.shape)
df.info()
print(df.isna().sum())
print('baris duplikat:', df.duplicated().sum())
print('motor_id unik:', df['motor_id'].nunique())

(79652, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79652 entries, 0 to 79651
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   motor_id          79652 non-null  object 
 1   tipe_motor        79652 non-null  object 
 2   lokasi            79652 non-null  object 
 3   hari_operasi      79652 non-null  int64  
 4   beban_pct         79652 non-null  float64
 5   arus_A            78050 non-null  float64
 6   tegangan_V        79652 non-null  float64
 7   suhu_belitan_C    78858 non-null  float64
 8   getaran_rms_mm_s  79652 non-null  float64
 9   kecepatan_rpm     79652 non-null  float64
 10  rul_hari          79652 non-null  int64  
dtypes: float64(6), int64(2), object(3)
memory usage: 6.7+ MB
motor_id               0
tipe_motor             0
lokasi                 0
hari_operasi           0
beban_pct              0
arus_A              1602
tegangan_V             0
suhu_belitan_C       794
getara

In [ ]:
import numpy as np
import pandas as pd

KOLOM_SENSOR = ['beban_pct', 'arus_A', 'tegangan_V',
                'suhu_belitan_C', 'getaran_rms_mm_s', 'kecepatan_rpm']

def rapikan_id(s):
    angka = ''.join(ch for ch in str(s) if ch.isdigit())
    return f'M-{int(angka):03d}'

def rapikan_tipe(s):
    s = str(s).upper().replace(' ', '').replace(',', '.')
    return s.replace('KW', 'kW')

def bersihkan_data(df):
    df = df.copy()
    df['motor_id'] = df['motor_id'].map(rapikan_id)
    df['tipe_motor'] = df['tipe_motor'].map(rapikan_tipe)
    # tipe yang salah di beberapa baris: ambil yang paling sering per motor
    df['tipe_motor'] = df.groupby('motor_id')['tipe_motor'].transform(lambda s: s.mode().iloc[0])
    df = df.drop_duplicates()
    # suhu: nilai > 150 adalah Kelvin
    df['suhu_belitan_C'] = np.where(df['suhu_belitan_C'] > 150,
                                    df['suhu_belitan_C'] - 273.15,
                                    df['suhu_belitan_C'])
    # getaran <= 0 tidak mungkin: kode error sensor
    df.loc[df['getaran_rms_mm_s'] <= 0, 'getaran_rms_mm_s'] = np.nan
    df = df.sort_values(['motor_id', 'hari_operasi']).reset_index(drop=True)
    # isi nilai kosong hanya dari data motor yang sama
    for c in ['arus_A', 'suhu_belitan_C', 'getaran_rms_mm_s']:
        df[c] = df.groupby('motor_id')[c].transform(lambda s: s.interpolate(limit_direction='both'))
    return df

bersih = bersihkan_data(df)

In [ ]:
assert bersih['motor_id'].nunique() == 300
assert bersih.groupby('motor_id')['tipe_motor'].nunique().max() == 1
assert bersih[KOLOM_SENSOR].isna().sum().sum() == 0
assert bersih['suhu_belitan_C'].between(20, 150).all()
assert not bersih.duplicated(['motor_id', 'hari_operasi']).any()
print('Semua pemeriksaan lolos!', bersih.shape)

Semua pemeriksaan lolos! (79159, 11)


In [ ]:
bersih.to_csv('clean.csv', index=False)

In [ ]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

url = 'https://raw.githubusercontent.com/dickypps/mkb-ml-rul-motor/main/data/clean.csv'
df = pd.read_csv(url)

# bagi 1: pisahkan test (20% motor)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
idx_tv, idx_test = next(gss.split(df, groups=df['motor_id']))
trainval, test = df.iloc[idx_tv], df.iloc[idx_test]

# bagi 2: dari sisanya, pisahkan val (25% dari sisa = 20% dari total)
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
idx_tr, idx_va = next(gss2.split(trainval, groups=trainval['motor_id']))
train, val = trainval.iloc[idx_tr], trainval.iloc[idx_va]

# BUKTI: tidak ada motor yang muncul di dua kelompok
assert set(train['motor_id']).isdisjoint(val['motor_id'])
assert set(train['motor_id']).isdisjoint(test['motor_id'])
assert set(val['motor_id']).isdisjoint(test['motor_id'])

print('jumlah motor  -> train:', train['motor_id'].nunique(),
      '| val:', val['motor_id'].nunique(),
      '| test:', test['motor_id'].nunique())
print('jumlah baris  -> train:', len(train), '| val:', len(val), '| test:', len(test))

jumlah motor  -> train: 180 | val: 60 | test: 60
jumlah baris  -> train: 47365 | val: 15337 | test: 16457


In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

KOLOM_SENSOR = ['beban_pct', 'arus_A', 'tegangan_V',
                'suhu_belitan_C', 'getaran_rms_mm_s', 'kecepatan_rpm']

def laporan_mae(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    kritis = y_true <= 60          # motor yang hampir rusak
    return {'mae_semua': float(mean_absolute_error(y_true, y_pred)),
            'mae_rul_le_60': float(mean_absolute_error(y_true[kritis], y_pred[kritis]))}

# latih baseline dengan fitur sensor asli
base = LinearRegression().fit(train[KOLOM_SENSOR], train['rul_hari'])

# ukur di data validasi
hasil = laporan_mae(val['rul_hari'], base.predict(val[KOLOM_SENSOR]))
print('Baseline regresi linear (val):')
print('  MAE semua baris      :', round(hasil['mae_semua'], 2), 'hari')
print('  MAE rul_hari <= 60   :', round(hasil['mae_rul_le_60'], 2), 'hari')

Baseline regresi linear (val):
  MAE semua baris      : 47.36 hari
  MAE rul_hari <= 60   : 29.88 hari




 baseline meleset rata-rata 29,88 hari untuk motor yang hampir rusak. Itu besar, karena pengiriman motor pengganti cuma 14-28 hari. Penyebabnya, hubungan antara sensor dan sisa umur tidak lurus, sedangkan regresi linear hanya bisa menarik garis lurus. Jadi ada banyak ruang untuk mengalahkannya.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor

def buat_fitur(df):
    df = df.copy()
    df['arus_per_beban'] = df['arus_A'] / df['beban_pct'].clip(lower=1)
    df['getaran_x_suhu'] = df['getaran_rms_mm_s'] * df['suhu_belitan_C']
    return df

FITUR_MODEL = KOLOM_SENSOR + ['arus_per_beban', 'getaran_x_suhu']
X_COLS = FITUR_MODEL + ['tipe_motor']

def buat_model(**kw):
    prep = ColumnTransformer(
        [('tipe', OneHotEncoder(handle_unknown='ignore'), ['tipe_motor'])],
        remainder='passthrough')
    gbr = HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05,
                                        random_state=42, **kw)
    return Pipeline([('prep', prep), ('gbr', gbr)])

tr, va = buat_fitur(train), buat_fitur(val)
model = buat_model().fit(tr[X_COLS], tr['rul_hari'])
hasil = laporan_mae(va['rul_hari'], model.predict(va[X_COLS]))

print('Gradient boosting (val):')
print('  MAE semua baris      :', round(hasil['mae_semua'], 2), 'hari')
print('  MAE rul_hari <= 60   :', round(hasil['mae_rul_le_60'], 2), 'hari')

Gradient boosting (val):
  MAE semua baris      : 42.58 hari
  MAE rul_hari <= 60   : 15.38 hari


- Untuk motor yang hampir rusak (metrik utama), tebakan meleset turun dari ±30 hari menjadi ±15 hari, atau sekitar 49% lebih akurat.
- MAE semua baris masih besar (42 hari). Kemungkinan penyebabnya, motor yang masih baru terlihat sehat dan sensornya hampir sama, jadi sisa umurnya yang masih panjang sulit ditebak. Ini bagus dituliskan di bagian keterbatasan laporan. Karena klien lebih peduli motor yang hampir rusak, ini bukan masalah besar.

In [ ]:
def coba(nama, model, cols, y_train):
    model.fit(tr[cols], y_train)
    h = laporan_mae(va['rul_hari'], model.predict(va[cols]))
    print(f'{nama:<38} semua={h["mae_semua"]:.2f}   <=60={h["mae_rul_le_60"]:.2f}')

y = tr['rul_hari']

coba('1. GBR awal (acuan)',               buat_model(),                       X_COLS, y)
coba('2. loss absolute_error',            buat_model(loss='absolute_error'),  X_COLS, y)
coba('3. target dipotong di 150 hari',    buat_model(),                       X_COLS, np.minimum(y, 150))
coba('4. + hari_operasi',                 buat_model(),                       X_COLS + ['hari_operasi'], y)
coba('5. tanpa fitur turunan',            buat_model(),                       KOLOM_SENSOR + ['tipe_motor'], y)

1. GBR awal (acuan)                    semua=42.58   <=60=15.38
2. loss absolute_error                 semua=42.31   <=60=13.86
3. target dipotong di 150 hari         semua=52.97   <=60=14.46
4. + hari_operasi                      semua=40.20   <=60=10.44
5. tanpa fitur turunan                 semua=42.66   <=60=15.44


In [ ]:
X4 = X_COLS + ['hari_operasi']

coba('6. absolute_error + hari_operasi',  buat_model(loss='absolute_error'), X4, y)
coba('7. ... + max_depth=6',              buat_model(loss='absolute_error', max_depth=6), X4, y)
coba('8. ... + min_samples_leaf=50',      buat_model(loss='absolute_error', min_samples_leaf=50), X4, y)

6. absolute_error + hari_operasi       semua=40.13   <=60=9.22
7. ... + max_depth=6                   semua=40.17   <=60=9.23
8. ... + min_samples_leaf=50           semua=40.18   <=60=9.14


In [ ]:
# fitur final
X_FINAL = X_COLS + ['hari_operasi']

tv, te = buat_fitur(trainval), buat_fitur(test)

# model final
final = buat_model(loss='absolute_error').fit(tv[X_FINAL], tv['rul_hari'])
hasil_test = laporan_mae(te['rul_hari'], final.predict(te[X_FINAL]))

# baseline, dilatih dengan data yang sama, untuk perbandingan yang adil
base_final = LinearRegression().fit(trainval[KOLOM_SENSOR], trainval['rul_hari'])
hasil_base = laporan_mae(test['rul_hari'], base_final.predict(test[KOLOM_SENSOR]))

print('HASIL TEST (dibuka satu kali)')
print('  Baseline : semua =', round(hasil_base['mae_semua'], 2),
      '| <=60 =', round(hasil_base['mae_rul_le_60'], 2))
print('  Model    : semua =', round(hasil_test['mae_semua'], 2),
      '| <=60 =', round(hasil_test['mae_rul_le_60'], 2))

HASIL TEST (dibuka satu kali)
  Baseline : semua = 54.9 | <=60 = 30.46
  Model    : semua = 48.66 | <=60 = 8.77


- Untuk motor yang hampir rusak, tebakan model rata-rata meleset 8,77 hari, turun dari 30,46 hari (sekitar 71% lebih akurat). Itu di bawah waktu pengiriman minimum motor pengganti (14 hari).
- Angka test (8,77) dekat dengan validasi (9,22), jadi modelnya stabil dan tidak sekadar hafal data.
- MAE semua baris masih tinggi (48,66). Motor yang masih baru sulit ditebak sisa umurnya. Tulis ini jujur di bagian keterbatasan laporan.
- Mulai sekarang jangan ubah model berdasarkan angka test.

In [ ]:
import joblib, json, os, sklearn

os.makedirs('models', exist_ok=True)
joblib.dump(final, 'models/model.joblib')

FITUR_BATAS = KOLOM_SENSOR + ['hari_operasi']
batas = {c: [float(trainval[c].min()), float(trainval[c].max())] for c in FITUR_BATAS}

metadata = {
    'versi': '1.0',
    'fitur': X_FINAL,
    'target': 'rul_hari',
    'batas_data_latih': batas,
    'ambang_status': {'segera_ganti_hari': 14, 'pesan_sekarang_hari': 45},
    'metrik_test': hasil_test,
    'metrik_baseline_test': hasil_base,
    'versi_sklearn': sklearn.__version__,
}
json.dump(metadata, open('models/metadata.json', 'w'), indent=2)
print('scikit-learn versi:', sklearn.__version__)

from google.colab import files
files.download('models/model.joblib')
files.download('models/metadata.json')

scikit-learn versi: 1.6.1


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>